In [84]:
"""
annotate_and_push.py

1. Pulls every image (reference photos + frames) from the HF dataset repo.
2. Runs SAM3 (text-prompted "QR Code") on each one to get bounding boxes.
3. Pushes the boxes back to the SAME repo as new annotation files, tagged
   with qr_id / session_id / source so each box can never be mismatched to
   the wrong QR code or video. The original images are left untouched.

Paste into ONE Kaggle cell and run (Internet ON, GPU on, HF_TOKEN secret
attached), or run locally with `python scripts/annotate_and_push.py` from a
venv that has a .env file with HF_TOKEN set and a CUDA GPU available
(SAM3 on CPU is very slow for hundreds of frames).

Repo files this ADDS:
    annotations/reference_photos.jsonl      one row per reference photo
    annotations/frames/{session_id}.jsonl   one row per frame, per session
    annotations/all_boxes.jsonl             everything in one file (convenience)

"""

'\nannotate_and_push.py\n\n1. Pulls every image (reference photos + frames) from the HF dataset repo.\n2. Runs SAM3 (text-prompted "QR Code") on each one to get bounding boxes.\n3. Pushes the boxes back to the SAME repo as new annotation files, tagged\n   with qr_id / session_id / source so each box can never be mismatched to\n   the wrong QR code or video. The original images are left untouched.\n\nPaste into ONE Kaggle cell and run (Internet ON, GPU on, HF_TOKEN secret\nattached), or run locally with `python scripts/annotate_and_push.py` from a\nvenv that has a .env file with HF_TOKEN set and a CUDA GPU available\n(SAM3 on CPU is very slow for hundreds of frames).\n\nRepo files this ADDS:\n    annotations/reference_photos.jsonl      one row per reference photo\n    annotations/frames/{session_id}.jsonl   one row per frame, per session\n    annotations/all_boxes.jsonl             everything in one file (convenience)\n\n'

In [85]:
import glob
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import CommitOperationAdd, HfApi, snapshot_download
from PIL import Image
from huggingface_hub import login

In [86]:
#CONFIG
REPO_ID = "RakshitAi/IITB-BuggyQR"
LOCAL_DIR = "/kaggle/working/buggyqr_data"    
OUT_DIR = "/kaggle/working/qr_annotations"      
SAM3_THRESHOLD = 0.5
SAM3_MASK_THRESHOLD = 0.5
COMMIT_MESSAGE = "Add SAM3 QR bounding box annotations"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [87]:
#Token: Kaggle secret first, then .env / environment
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("No HF_TOKEN found (Kaggle secret, .env, or environment variable).")
    login(token=token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = token  
    return token


def get_sam3_token():
    """
    Separate token for loading the gated facebook/sam3 model, in case your
    HF_TOKEN is a fine-grained token scoped only to your dataset repo.
    Falls back to HF_TOKEN if no SAM3_HF_TOKEN is set (most setups only need one).
    """
    return _get_secret_or_env("SAM3_HF_TOKEN") or _get_secret_or_env("HF_TOKEN")
 
 
def _get_secret_or_env(name):
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    return os.environ.get(name)

In [88]:
# 1. Pull every image from the dataset, keeping repo-relative paths
def pull_dataset_images(repo_id=REPO_ID, local_dir=LOCAL_DIR, token=None):
    """
    Downloads raw/reference_photos/* and frames/**/*.jpg and returns a list of dicts:
        {"local_path", "repo_path", "qr_id", "session_id", "source"}
    repo_path is the exact path the image lives at in the HF repo, used later
    to tag each annotation back to its source image unambiguously.
    """
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=local_dir,
        allow_patterns=["raw/reference_photos/*", "frames/**/*.jpg"],
        token=token,
    )

    items = []
    for p in sorted(glob.glob(os.path.join(local_dir, "raw", "reference_photos", "*"))):
        if p.lower().endswith((".jpg", ".jpeg", ".png")):
            qr_id = os.path.splitext(os.path.basename(p))[0]
            items.append({
                "local_path": p,
                "repo_path": f"raw/reference_photos/{os.path.basename(p)}",
                "qr_id": qr_id,
                "session_id": None,
                "source": "reference",
            })

    for sdir in sorted(glob.glob(os.path.join(local_dir, "frames", "*"))):
        session_id = os.path.basename(sdir)
        qr_id = session_id.rsplit("_", 2)[0]  # strip _<date>_<take>
        for p in sorted(glob.glob(os.path.join(sdir, "*.jpg"))):
            items.append({
                "local_path": p,
                "repo_path": f"frames/{session_id}/{os.path.basename(p)}",
                "qr_id": qr_id,
                "session_id": session_id,
                "source": "frame",
            })

    n_ref = sum(i["source"] == "reference" for i in items)
    n_frm = sum(i["source"] == "frame" for i in items)
    print(f"Pulled {len(items)} images ({n_ref} reference, {n_frm} frames)")
    return items

In [89]:
# 2. SAM3 QR detection -> bounding boxes
_sam3_model = None
_sam3_processor = None

In [90]:
def _load_sam3(token=None):
    global _sam3_model, _sam3_processor
    if _sam3_model is None:
        from transformers import Sam3Model, Sam3Processor
        _sam3_model = Sam3Model.from_pretrained("facebook/sam3", token=token).to(device)
        _sam3_processor = Sam3Processor.from_pretrained("facebook/sam3", token=token)
    return _sam3_model, _sam3_processor
 
 
def warm_up_sam3(items, sam3_token=None):
    """
    Runs SAM3 once on a throwaway image so model download + weight load +
    CUDA init happen here, not inside the first timed/pushed image.
    """
    if not items:
        return
    print("Warming up SAM3 (downloading/loading weights, priming CUDA)...")
    t0 = time.time()
    _load_sam3(token=sam3_token)
    _ = detect_qr_boxes(Image.open(items[0]["local_path"]).convert("RGB"))
    print(f"  done in {time.time() - t0:.1f}s (one-time cost)")

In [91]:
def detect_qr_boxes(image: Image.Image, threshold=SAM3_THRESHOLD, mask_threshold=SAM3_MASK_THRESHOLD):
    """
    Runs SAM3 with the text prompt "QR Code" and returns a list of box dicts:
        {"box_xyxy": [x1,y1,x2,y2], "box_xyxyn": [x1,y1,x2,y2] normalized 0-1, "score": float}
    box_xyxyn lets you reuse the boxes even if images get resized later.
    """
    model, processor = _load_sam3()
    W, H = image.size

    inputs = processor(images=image, text="QR Code", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    res = processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    def to_np(x):
        return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)

    raw_boxes = to_np(res["boxes"]).reshape(-1, 4)
    scores = to_np(res["scores"]).reshape(-1)

    boxes = []
    for (x1, y1, x2, y2), s in zip(raw_boxes, scores):
        boxes.append({
            "box_xyxy": [round(float(x1), 1), round(float(y1), 1), round(float(x2), 1), round(float(y2), 1)],
            "box_xyxyn": [round(float(x1 / W), 5), round(float(y1 / H), 5), round(float(x2 / W), 5), round(float(y2 / H), 5)],
            "score": round(float(s), 4),
        })
    return boxes

In [92]:
# 3. Annotate every pulled image
def annotate(items):
    """Returns a list of annotation records, one per image."""
    records = []
    t_start = time.time()
    for i, item in enumerate(items):
        try:
            image = Image.open(item["local_path"]).convert("RGB")
        except Exception as e:
            print(f"  [skip] could not open {item['local_path']}: {e}")
            continue

        W, H = image.size
        boxes = detect_qr_boxes(image)

        records.append({
            "file_name": item["repo_path"],     # exact path in the HF repo -> unambiguous image<->box link
            "qr_id": item["qr_id"],
            "session_id": item["session_id"],
            "source": item["source"],
            "image_width": W,
            "image_height": H,
            "detector": "sam3",
            "num_boxes": len(boxes),
            "boxes": boxes,
        })

        if (i + 1) % 50 == 0 or (i + 1) == len(items):
            elapsed = time.time() - t_start
            print(f"  annotated {i + 1}/{len(items)}  ({elapsed:.1f}s elapsed)")

    n_found = sum(r["num_boxes"] > 0 for r in records)
    print(f"QR found in {n_found}/{len(records)} images")
    return records

In [93]:
# 4. Write annotation files locally, matching the repo's session structure
def write_annotation_files(records, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    (out_dir / "frames").mkdir(parents=True, exist_ok=True)

    ref_records = [r for r in records if r["source"] == "reference"]
    with open(out_dir / "reference_photos.jsonl", "w") as f:
        for r in ref_records:
            f.write(json.dumps(r) + "\n")

    by_session = {}
    for r in records:
        if r["source"] == "frame":
            by_session.setdefault(r["session_id"], []).append(r)
    for session_id, recs in by_session.items():
        with open(out_dir / "frames" / f"{session_id}.jsonl", "w") as f:
            for r in recs:
                f.write(json.dumps(r) + "\n")

    with open(out_dir / "all_boxes.jsonl", "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")

    print(f"Wrote {len(ref_records)} reference + {len(records) - len(ref_records)} frame "
          f"annotation rows across {1 + len(by_session) + 1} files -> {out_dir}")
    return out_dir

In [94]:
# 5. Push annotation files back to the SAME HF repo, in one commit
def push_annotations(out_dir, repo_id=REPO_ID, token=None):
    api = HfApi(token=token)
    ops = []
    for local_path in Path(out_dir).rglob("*.jsonl"):
        repo_path = "annotations/" + str(local_path.relative_to(out_dir)).replace(os.sep, "/")
        ops.append(CommitOperationAdd(path_in_repo=repo_path, path_or_fileobj=str(local_path)))

    api.create_commit(
        repo_id=repo_id,
        repo_type="dataset",
        operations=ops,
        commit_message=COMMIT_MESSAGE,
    )
    print(f"Pushed {len(ops)} annotation files to {repo_id}/annotations/")

In [95]:
token = get_hf_token()
sam3_token = get_sam3_token()
print("Pulling dataset...")
items = pull_dataset_images(token=token)

warm_up_sam3(items, sam3_token=sam3_token)

print("\nDetecting QR bounding boxes with SAM3...")
records = annotate(items)

print("\nWriting annotation files...")
out_dir = write_annotation_files(records)

print("\nPushing annotations to Hugging Face...")
push_annotations(out_dir, token=token)

Pulling dataset...


Fetching 363 files:   0%|          | 0/363 [00:00<?, ?it/s]

Pulled 363 images (1 reference, 362 frames)
Warming up SAM3 (downloading/loading weights, priming CUDA)...


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

  done in 8.3s (one-time cost)

Detecting QR bounding boxes with SAM3...
  annotated 50/363  (122.3s elapsed)
  annotated 100/363  (242.5s elapsed)
  annotated 150/363  (362.6s elapsed)
  annotated 200/363  (483.3s elapsed)
  annotated 250/363  (603.7s elapsed)
  annotated 300/363  (724.2s elapsed)
  annotated 350/363  (844.7s elapsed)


No files have been modified since last commit. Skipping to prevent empty commit.


  annotated 363/363  (875.8s elapsed)
QR found in 363/363 images

Writing annotation files...
Wrote 1 reference + 362 frame annotation rows across 3 files -> /kaggle/working/qr_annotations

Pushing annotations to Hugging Face...
Pushed 3 annotation files to RakshitAi/IITB-BuggyQR/annotations/
